In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Tabel Tipe A

PNS

In [ ]:
import os

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/A/PNS'
file_listA1 = os.listdir(folder_path)

print(f"Files and directories in '{folder_path}':")
for item in file_listA1:
    print(item)

Files and directories in '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/A/PNS':
Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.csv
Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2022.csv
Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2021.csv
Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2020.csv
Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2019.csv
Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2018.csv
Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2017.csv
Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.xlsx
Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabup

In [ ]:
import pandas as pd

for file_name in file_listA1:
    file_path = os.path.join(folder_path, file_name)

    # cek hanya file csv
    if file_name.endswith(".csv"):
        try:
            # baca csv (handle delimiter)
            try:
                df = pd.read_csv(file_path)
            except:
                df = pd.read_csv(file_path, sep=';')

            # nama file baru (.xlsx)
            new_name = os.path.splitext(file_name)[0] + ".xlsx"
            new_path = os.path.join(folder_path, new_name)

            # save ke xlsx
            df.to_excel(new_path, index=False)

            print(f"Converted: {file_name} → {new_name}")

        except Exception as e:
            print(f"Gagal convert {file_name}: {e}")

Converted: Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.csv → Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.xlsx
Converted: Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2022.csv → Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2022.xlsx
Converted: Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2021.csv → Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2021.xlsx
Converted: Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2020.csv → Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2020.xlsx
Converted: Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2019.csv → Juml

In [ ]:
import pandas as pd
import numpy as np
import os

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/A/PNS'
file_listA1 = os.listdir(folder_path)

file_listA1_cleaned = []

def preprocess_A1(file_path):

    df = pd.read_excel(file_path)

    # =========================
    # 1. BERSIHKAN KOLOM A
    # =========================
    df.iloc[:,0] = (
        df.iloc[:,0]
        .astype(str)
        .str.replace(r"\n.*", "", regex=True)   # hapus newline dan teks setelahnya
        .str.split('/')                         # ambil sebelum /
        .str[0]
        .str.strip()
    )

    # =========================
    # 2. HAPUS BARIS YANG ADA '...'
    # =========================
    df = df[~df.astype(str).apply(lambda x: x.str.contains(r"\.\.\.")).any(axis=1)]

    # =========================
    # 3. GANTI '-' MENJADI 0
    # =========================
    df = df.replace('-', 0)
    df = df.replace('', 0)
    df = df.fillna(0)

    # =========================
    # 4. HAPUS KOLOM PALING KANAN
    # =========================
    df = df.iloc[:, :-1]

    # =========================
    # BERSIHKAN NAMA KOLOM
    # =========================
    new_cols = []

    for col in df.columns:

        col = str(col)

        if "Perempuan" in col:
            new_cols.append("Perempuan")
        elif "Laki" in col:
            new_cols.append("Laki-Laki")
        else:
            new_cols.append(col)

    df.columns = new_cols

    # =========================
    # HAPUS BARIS 'JUMLAH' DAN SETERUSNYA
    # =========================
    col0 = (
        df.iloc[:,0]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    jumlah_idx = col0.str.contains("jumlah", case=False, na=False)

    if jumlah_idx.any():
        df = df.loc[:jumlah_idx[jumlah_idx].index[0]-1]

    return df


for f in file_listA1:

    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):

            df_clean = preprocess_A1(path)

            file_listA1_cleaned.append((f, df_clean))

            print(f"Sukses: {f}")

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listA1_cleaned)}")

Sukses: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.xlsx
Sukses: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2022.xlsx
Sukses: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2021.xlsx
Sukses: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2020.xlsx
Sukses: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2019.xlsx
Sukses: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2018.xlsx
Sukses: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2017.xlsx
Sukses: Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.xlsx
Sukses: Jumlah Pegawai Nege

Cek Duplikat

In [ ]:
import re
from collections import defaultdict

def normalize_filename(filename):
    temp = filename

    # hapus "di Kabupaten Buleleng"
    temp = re.sub(r"di Kabupaten Buleleng", "", temp, flags=re.IGNORECASE)

    # hapus teks dalam tanda kurung (termasuk kurungnya)
    temp = re.sub(r"\(.*?\)", "", temp)

    # rapikan spasi
    temp = " ".join(temp.split())

    return temp.strip()

# grouping berdasarkan nama yang sudah dinormalisasi
groups = defaultdict(list)

for f in file_listA1:
    norm = normalize_filename(f)
    groups[norm].append(f)

# cek dan tampilkan hasil
found_duplicate = False

for norm, files in groups.items():
    if len(files) > 1:
        if not found_duplicate:
            print("Duplikat ditemukan:\n")
        found_duplicate = True

        print(f"Group: {norm}")
        for f in files:
            print(f"- {f}")
        print()

if not found_duplicate:
    print("Tidak ada duplikat.")

Tidak ada duplikat.


In [ ]:
file_listA1_cleaned[0]

('Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.xlsx',
                Tingkat Pendidikan  Laki-Laki  Perempuan
 0              Sekolah Dasar (SD)       25.0        8.0
 1  Sekolah Menengah Pertama (SMP)       48.0        7.0
 2     Sekolah Menengah Atas (SMA)      649.0      250.0
 3                       Diploma I        5.0        4.0
 4                      Diploma II       89.0       69.0
 5                     Diploma III      181.0      530.0
 6                      Diploma IV       28.0       44.0
 7                              S1     1830.0     2044.0
 8                              S2      290.0      265.0
 9                              S3        1.0        1.0)

In [ ]:
output_folder = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/A_cleaned1'

for f, df in file_listA1_cleaned:
    # ganti ekstensi jadi .xlsx
    new_name = os.path.splitext(f)[0] + ".xlsx"
    output_path = os.path.join(output_folder, new_name)

    df.to_excel(output_path, index=False)

    print(f"Saved: {new_name}")

Saved: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.xlsx
Saved: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2022.xlsx
Saved: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2021.xlsx
Saved: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2020.xlsx
Saved: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2019.xlsx
Saved: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2018.xlsx
Saved: Jumlah Pegawai Negeri Sipil Menurut Tingkat Pendidikan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2017.xlsx
Saved: Jumlah Pegawai Negeri Sipil Menurut Jabatan dan Jenis Kelamin di Kabupaten Buleleng, Desember 2023.xlsx
Saved: Jumlah Pegawai Negeri Sipil 

Time Series Jumlah PNS

In [ ]:
import pandas as pd
import os
import re

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/A/PNS'
file_listA1 = os.listdir(folder_path)

data_ts = []

for f in file_listA1:

    if not f.endswith(".xlsx"):
        continue

    path = os.path.join(folder_path, f)

    df = pd.read_excel(path)

    # =========================
    # AMBIL BARIS JUMLAH
    # =========================
    jumlah_row = df[
        df.iloc[:,0].astype(str).str.contains("Jumlah", case=False, na=False)
    ]

    if len(jumlah_row) == 0:
        continue

    # =========================
    # AMBIL NILAI PALING KANAN
    # =========================
    value = jumlah_row.iloc[0, -1]

    # =========================
    # AMBIL TAHUN DARI NAMA FILE
    # =========================
    year_match = re.search(r'20\d{2}', f)

    if year_match:
        year = int(year_match.group())
    else:
        continue

    data_ts.append({
        "Tahun": year,
        "PNS": value
    })


# =========================
# BUAT DATAFRAME
# =========================
df_timeseries = pd.DataFrame(data_ts)

# =========================
# HAPUS DUPLIKAT (TAHUN DAN NILAI SAMA)
# =========================
df_timeseries = df_timeseries.drop_duplicates(subset=["Tahun"])

# =========================
# URUTKAN TAHUN
# =========================
df_timeseries = df_timeseries.sort_values("Tahun").reset_index(drop=True)

print(df_timeseries)

   Tahun     PNS
0   2017    8363
1   2018    7943
2   2019    7837
3   2020  7726.0
4   2021  7220.0
5   2022  6910.0
6   2023  6368.0


# Tabel Tipe A2

In [ ]:
import os

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/RAW/A/'
file_listA2 = os.listdir(folder_path)

print(f"Files and directories in '{folder_path}':")
for item in file_listA2:
    print(item)

Files and directories in '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/RAW/A/':
Produksi Buah–Buahan dan Sayuran Tahunan Menurut Jenis Tanaman di Kabupaten Buleleng, 2023.csv
Produksi Buah–Buahan dan Sayuran Tahunan Menurut Jenis Tanaman di Kabupaten Buleleng, 2022.csv
Produksi Buah–Buahan dan Sayuran Tahunan Menurut Jenis Tanaman di Kabupaten Buleleng, 2021.csv
Produksi Buah–Buahan dan Sayuran Tahunan Menurut Jenis Tanaman di Kabupaten Buleleng, 2020.csv
Produksi Buah–Buahan dan Sayuran Tahunan Menurut Jenis Tanaman di Kabupaten Buleleng, 2019.csv
Produksi Buah–Buahan dan Sayuran Tahunan Menurut Jenis Tanaman di Kabupaten Buleleng, 2018.csv
Produksi Tanaman Hias Menurut  Jenis Tanaman di Kabupaten Buleleng, 2024.csv
Produksi Tanaman Hias Menurut  Jenis Tanaman di Kabupaten Buleleng, 2023.csv
Produksi Tanaman Hias Menurut  Jenis Tanaman di Kabupaten Buleleng, 2022.csv
Produksi Tanaman Hias Menurut  Jenis Tanaman di Kabupaten Buleleng, 2021.csv
Produksi Tanaman Hias Menurut  Jenis T

In [ ]:
import os
import openpyxl

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/RAW/A/'
file_listA2 = os.listdir(folder_path)

files_with_br = []

for filename in file_listA2:
    if not filename.endswith('.xlsx'):
        continue
    file_path = os.path.join(folder_path, filename)
    try:
        wb = openpyxl.load_workbook(file_path, data_only=True)
        found = False
        for sheet in wb.worksheets:
            for row in sheet.iter_rows():
                for cell in row:
                    if cell.value and isinstance(cell.value, str) and '<br' in cell.value:
                        found = True
                        break
                if found:
                    break
            if found:
                break
        if found:
            files_with_br.append(filename)
    except Exception as e:
        print(f"Gagal baca {filename}: {e}")

print(f"\nFile yang mengandung '<br': {len(files_with_br)}")
for f in files_with_br:
    print(f"  - {f}")


File yang mengandung '<br': 0


In [ ]:
import pandas as pd
import os
import re

# Ensure we are using the correct folder_path for file_listA2
folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/RAW/A/'
file_listA2 = os.listdir(folder_path)

print("Checking for '<br>' within the content of original files from file_listA2:")

found_in_original_content = []

for filename in file_listA2:
    file_path = os.path.join(folder_path, filename)
    df = None

    try:
        if filename.endswith(('.csv', '.xlsx')):
            if filename.endswith('.csv'):
                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    # Try with semicolon delimiter if initial read fails
                    df = pd.read_csv(file_path, sep=';')
            elif filename.endswith('.xlsx'):
                df = pd.read_excel(file_path)

            if df is not None:
                # Convert all DataFrame content to string type to search for the substring
                df_str = df.astype(str)

                # Check if '<br' exists in any cell of the DataFrame
                if df_str.apply(lambda col: col.str.contains(r'<br', case=False, na=False)).any().any():
                    found_in_original_content.append(filename)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

if found_in_original_content:
    print("\nFiles from file_listA2 containing '<br>' in their content:")
    for f in found_in_original_content:
        print(f"- {f}")
else:
    print("\nNo files from file_listA2 found containing '<br>' in their content.")

In [ ]:
import numpy as np
import re
import os
import pandas as pd

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/A/'
file_listA2 = os.listdir(folder_path)

file_listA2_cleaned = []

def preprocess_A2(file_path, file_name):
    label_top = None
    df = pd.read_excel(file_path)
    cleaned_cols = []

    # =========================
    # BERSIHKAN NAMA KOLOM
    # =========================
    for col in df.columns:
        col = str(col)

        # handle <br/> dan versi encoded
        col = re.split(r'(<br\s*/?>|&lt;br\s*/?&gt;)', col)[0]

        # hapus sisa HTML kalau ada
        col = re.sub(r'&lt;.*?&gt;', '', col)

        # kalau ada beberapa (...) ambil yang terakhir saja
        matches = re.findall(r'\([^)]*\)', col)
        if len(matches) > 1:
            col = re.sub(r'\([^)]*\)', '', col).strip() + ' ' + matches[-1]

        cleaned_cols.append(col.strip())
    df.columns = cleaned_cols

    # =========================
    # BERSIHKAN KOLOM PERTAMA (UMUM)
    # =========================
    df.iloc[:,0] = (
        df.iloc[:,0]
        .astype(str)
        .str.split('\n').str[0]                 # ambil baris pertama saja
        .str.replace(r'/.*', '', regex=True)    # hapus '/' dan seterusnya
        .str.strip()
    )

    col2 = df.iloc[:,1]

    df = df[
        col2.notna() &
        (col2.astype(str).str.strip() != '') &
        (col2.astype(str).str.strip() != '...')
    ]

    # =========================
    # KHUSUS FILE TANAMAN
    # =========================
    if "tanaman" in file_name.lower():

        # hapus baris kosong atau '...'
        df = df[~df.astype(str).apply(lambda x: x.str.contains(r"\.\.\.")).any(axis=1)]
        df = df.replace('', np.nan).dropna(how='all')

        # ubah ke numerik (selain kolom pertama)
        for col in df.columns[1:]:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # hitung total per baris
        df['TOTAL'] = df.iloc[:,1:].sum(axis=1)

        # tentukan jumlah top
        n = 5 if len(df) >= 5 else 3
        label_top = f"{n} Besar"

        df = df.nlargest(n, 'TOTAL')

        df = df.drop(columns=['TOTAL'])

        # bersihkan tanda kurung (hapus / dalam kurung)
        df.iloc[:,0] = (
            df.iloc[:,0]
            .str.replace(r'\(([^)]*)/[^)]*\)', r'(\1)', regex=True)
        )

    # =========================
    # SELAIN TANAMAN
    # =========================
    else:
      # =========================
      # HAPUS BARIS PDRB DAN SETERUSNYA
      # =========================
      col0 = df.iloc[:,0].astype(str)

      pdrb_idx = col0.str.contains("Produk Domestik Regional Bruto", case=False, na=False)

      if pdrb_idx.any():
          df = df.loc[:pdrb_idx[pdrb_idx].index[0]-1]

      # =========================
      # PERBAIKI FORMAT KODE SEKTOR
      # =========================
      df.iloc[:,0] = (
          df.iloc[:,0]
          # kasus: APerdagangan → A. Perdagangan
          .str.replace(r'^([A-Z])([A-Za-z])', r'\1. \2', regex=True)
          # kasus: R,S,T,UJasa → R,S,T,U. Jasa
          .str.replace(r'^([A-Z](?:,[A-Z])*)([A-Za-z])', r'\1. \2', regex=True)
      )

    return df, label_top


# =========================
# LOOP FILE
# =========================
for f in file_listA2:

    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):

            df_clean, label_top = preprocess_A2(path, f)
            file_listA2_cleaned.append((f, df_clean, label_top))

            print(f"Sukses: {f}")

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listA2_cleaned)}")

Sukses: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2025.xlsx
Sukses: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2024.xlsx
Sukses: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2023.xlsx
Sukses: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2022.xlsx
Sukses: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2021.xlsx
Sukses: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2020.xlsx
Sukses: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut 

In [ ]:
file_listA2_cleaned[77]

('Produksi Perkebunan Rakyat Menurut Jenis Tanaman di Kabupaten Buleleng (Ribu ton), 2018.xlsx',
    Jenis Tanaman  Produksi Tanaman Perkebunan Rakyat (Ribu Ton)
 1         Kelapa                                            9.5
 4           Kopi                                            1.2
 12      Tembakau                                            0.7,
 '3 Besar')

In [ ]:
output_folder = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/A_cleaned2'

for f, df, label_top in file_listA2_cleaned:

    base_name = os.path.splitext(f)[0]

    if label_top:
        new_name = f"{label_top} {base_name}.xlsx"
    else:
        new_name = f"{base_name}.xlsx"

    output_path = os.path.join(output_folder, new_name)

    df.to_excel(output_path, index=False)

    print(f"Saved: {new_name}")

Saved: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2025.xlsx
Saved: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2024.xlsx
Saved: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2023.xlsx
Saved: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2022.xlsx
Saved: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2021.xlsx
Saved: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapangan Usaha di Kabupaten Buleleng (persen), 2020.xlsx
Saved: Laju Pertumbuhan Produk Domestik Regional Bruto Atas Dasar Harga Konstan 2010 Menurut Lapanga

# Tabel Tipe A3

In [ ]:
import os

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL'
file_listA3 = os.listdir(folder_path)

print(f"Files and directories in '{folder_path}':")
for item in file_listA3:
    print(item)

Files and directories in '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL':
Jaringan PLN di Wilayah Kabupaten Buleleng, 2013.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2012.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2011.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2010.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2009.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2008.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2007.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2006.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2005.csv
Jaringan PLN di Wilayah Kabupaten Buleleng, 2004.csv
A
B
C
D
E
B_cleaned
C_cleaned
D_cleaned
D_cleaned2
B_cleaned2
E_cleaned
A_cleaned1
A_cleaned2
Jaringan PLN di Wilayah Kabupaten Buleleng, 2013.xlsx
Jaringan PLN di Wilayah Kabupaten Buleleng, 2012.xlsx
Jaringan PLN di Wilayah Kabupaten Buleleng, 2011.xlsx
Jaringan PLN di Wilayah Kabupaten Buleleng, 2010.xlsx
Jaringan PLN di Wilayah Kabupaten Buleleng, 2009.xlsx
Jaringan PLN di Wilayah Kabup

In [ ]:
import pandas as pd

for file_name in file_listA3:
    file_path = os.path.join(folder_path, file_name)

    # cek hanya file csv
    if file_name.endswith(".csv"):
        try:
            # baca csv (handle delimiter)
            try:
                df = pd.read_csv(file_path)
            except:
                df = pd.read_csv(file_path, sep=';')

            # nama file baru (.xlsx)
            new_name = os.path.splitext(file_name)[0] + ".xlsx"
            new_path = os.path.join(folder_path, new_name)

            # save ke xlsx
            df.to_excel(new_path, index=False)

            print(f"Converted: {file_name} → {new_name}")

        except Exception as e:
            print(f"Gagal convert {file_name}: {e}")

Converted: Jaringan PLN di Wilayah Kabupaten Buleleng, 2013.csv → Jaringan PLN di Wilayah Kabupaten Buleleng, 2013.xlsx
Converted: Jaringan PLN di Wilayah Kabupaten Buleleng, 2012.csv → Jaringan PLN di Wilayah Kabupaten Buleleng, 2012.xlsx
Converted: Jaringan PLN di Wilayah Kabupaten Buleleng, 2011.csv → Jaringan PLN di Wilayah Kabupaten Buleleng, 2011.xlsx
Converted: Jaringan PLN di Wilayah Kabupaten Buleleng, 2010.csv → Jaringan PLN di Wilayah Kabupaten Buleleng, 2010.xlsx
Converted: Jaringan PLN di Wilayah Kabupaten Buleleng, 2009.csv → Jaringan PLN di Wilayah Kabupaten Buleleng, 2009.xlsx
Converted: Jaringan PLN di Wilayah Kabupaten Buleleng, 2008.csv → Jaringan PLN di Wilayah Kabupaten Buleleng, 2008.xlsx
Converted: Jaringan PLN di Wilayah Kabupaten Buleleng, 2007.csv → Jaringan PLN di Wilayah Kabupaten Buleleng, 2007.xlsx
Converted: Jaringan PLN di Wilayah Kabupaten Buleleng, 2006.csv → Jaringan PLN di Wilayah Kabupaten Buleleng, 2006.xlsx
Converted: Jaringan PLN di Wilayah Kabup

In [ ]:
import os
import pandas as pd

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/'
file_listA3 = os.listdir(folder_path)

file_listA3_cleaned = []

def preprocess_A3(file_path):

    df = pd.read_excel(file_path)

    # =========================
    # AMBIL BAGIAN DATA
    # =========================
    df_clean = df.iloc[2:, [0, 1]].copy()  # skip header atas

    df_clean.columns = ['Indikator', 'Nilai']

    # =========================
    # BERSIHKAN TEKS
    # =========================
    df_clean['Indikator'] = (
        df_clean['Indikator']
        .astype(str)
        .str.split('\n').str[0]   # kalau ada multi-line
        .str.replace(r'/.*', '', regex=True)  # hapus '/' dst
        .str.strip()
    )

    # =========================
    # KONVERSI NUMERIK
    # =========================
    df_clean['Nilai'] = pd.to_numeric(df_clean['Nilai'], errors='coerce')

    # =========================
    # HAPUS BARIS KOSONG
    # =========================
    df_clean = df_clean.dropna()

    return df_clean


# =========================
# LOOP FILE
# =========================
for f in file_listA3:

    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):

            df_clean = preprocess_A3(path)
            file_listA3_cleaned.append((f, df_clean))

            print(f"Sukses: {f}")

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listA3_cleaned)}")

Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2013.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2012.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2011.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2010.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2009.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2008.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2007.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2006.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2005.xlsx
Sukses: Jaringan PLN di Wilayah Kabupaten Buleleng, 2004.xlsx
Total berhasil: 10


In [ ]:
file_listA3_cleaned[0]

('Jaringan PLN di Wilayah Kabupaten Buleleng, 2013.xlsx',
               Indikator      Nilai
 2           JARINGAN TM    3971.31
 3              TIANG TM   60336.00
 4            TRAFO (BH)    1282.00
 5  DAYA TERPASANG (KVA)  152190.00)

In [ ]:
output_folder = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/A_cleaned3'

for f, df in file_listA3_cleaned:

    base_name = os.path.splitext(f)[0]
    new_name = f"{base_name}.xlsx"

    output_path = os.path.join(output_folder, new_name)

    df.to_excel(output_path, index=False)

    print(f"Saved: {new_name}")

Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2013.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2012.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2011.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2010.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2009.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2008.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2007.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2006.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2005.xlsx
Saved: Jaringan PLN di Wilayah Kabupaten Buleleng, 2004.xlsx


# Tabel Tipe B

In [ ]:
import os

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/CLEANED/B_cleaned'
file_listB = os.listdir(folder_path)

print(f"Files and directories in '{folder_path}':")
for item in file_listB:
    print(item)

Files and directories in '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/CLEANED/B_cleaned':
PDRB Seri 2010 Atas Dasar Harga Konstan Menurut Pengeluaran Kabupaten Buleleng, 2024.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2025.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2024.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2023.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2022.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2021.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2020.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2019.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2018.xlsx
Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2017.xlsx
Laju Indeks Im

In [ ]:
from openpyxl import load_workbook
import pandas as pd
import os

def preprocess_excel(file_path):
    wb = load_workbook(file_path)
    ws = wb.active

    # =========================
    # 1. CONVERT KE DATAFRAME
    # =========================
    df = pd.DataFrame(ws.values)

    # set header dari baris pertama
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)

    # =========================
    # 2. HAPUS BARIS TERAKHIR JIKA PDRB
    # =========================
    if len(df) > 0:
        last_row_text = str(df.iloc[-1, 0]).strip().lower()

        if ("pdrb" in last_row_text) or ("produk domestik regional bruto" in last_row_text):
            df = df.iloc[:-1].reset_index(drop=True)

    return df


# =========================
# LOOP FILE
# =========================
file_listB_cleaned = []

for f in file_listB:
    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):
            df = preprocess_excel(path)
            file_listB_cleaned.append((f, df))

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listB_cleaned)}")

Total berhasil: 125


In [ ]:
file_listB_cleaned[0]

('Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2016.xlsx',
 0                             Kategori Lapangan Usaha  \
 0              A. Pertanian, Kehutanan, dan Perikanan   
 1                      B. Pertambangan dan Penggalian   
 2                              C. Industri Pengolahan   
 3                        D. Pengadaan Listrik dan Gas   
 4   E. Pengadaan Air, Pengelolaan Sampah, Limbah d...   
 5                                       F. Konstruksi   
 6   G. Perdagangan Besar dan Eceran; Reparasi Mobi...   
 7                     H. Tranposrtasi dan Pergudangan   
 8             I. Penyediaan Akomodasi dan Makan Minum   
 9                         J. Informasi dan Komunikasi   
 10                      K. Jasa Keuangan dan Asuransi   
 11                                     L. Real Estate   
 12                               M,N. Jasa Perusahaan   
 13  O. Administrasi Pemerintahan, Pertahanan, dan ...   
 14                                 P

In [ ]:
import pandas as pd

for file_name in file_listB:
    file_path = os.path.join(folder_path, file_name)

    # cek hanya file csv
    if file_name.endswith(".csv"):
        try:
            # baca csv (handle delimiter)
            try:
                df = pd.read_csv(file_path)
            except:
                df = pd.read_csv(file_path, sep=';')

            # nama file baru (.xlsx)
            new_name = os.path.splitext(file_name)[0] + ".xlsx"
            new_path = os.path.join(folder_path, new_name)

            # save ke xlsx
            df.to_excel(new_path, index=False)

            print(f"Converted: {file_name} → {new_name}")

        except Exception as e:
            print(f"Gagal convert {file_name}: {e}")

Cek Duplikat

In [ ]:
import re
from collections import defaultdict

def normalize_filename(filename):
    temp = filename

    # hapus "di Kabupaten Buleleng"
    temp = re.sub(r"Kabupaten Buleleng", "", temp, flags=re.IGNORECASE)

    # hapus teks dalam tanda kurung (termasuk kurungnya)
    temp = re.sub(r"\(.*?\)", "", temp)

    # rapikan spasi
    temp = " ".join(temp.split())

    return temp.strip()

# grouping berdasarkan nama yang sudah dinormalisasi
groups = defaultdict(list)

for f in file_listB:
    norm = normalize_filename(f)
    groups[norm].append(f)

# cek dan tampilkan hasil
found_duplicate = False

for norm, files in groups.items():
    if len(files) > 1:
        if not found_duplicate:
            print("Duplikat ditemukan:\n")
        found_duplicate = True

        print(f"Group: {norm}")
        for f in files:
            print(f"- {f}")
        print()

if not found_duplicate:
    print("Tidak ada duplikat.")

Tidak ada duplikat.


In [ ]:
from openpyxl import load_workbook

def preprocess_excel(file_path):
    wb = load_workbook(file_path)
    ws = wb.active

    # =========================
    # 1. UNMERGE + FILL
    # =========================
    merged_cells = list(ws.merged_cells.ranges)

    for merged in merged_cells:
        min_row, min_col, max_row, max_col = (
            merged.min_row, merged.min_col,
            merged.max_row, merged.max_col
        )

        value = ws.cell(row=min_row, column=min_col).value

        ws.unmerge_cells(str(merged))

        for r in range(min_row, max_row + 1):
            for c in range(min_col, max_col + 1):
                ws.cell(row=r, column=c).value = value

    # =========================
    # 2. GABUNG B2 + B3
    # =========================
    b2 = ws["B2"].value or ""
    b3 = ws["B3"].value or ""

    # kalau float → ubah ke int dulu
    if isinstance(b3, float):
        b3 = int(b3)

    b3 = str(b3)

    ws["B2"] = f"{b2} {b3}".strip()
    ws["B3"] = None

    # =========================
    # 3. CONVERT KE DATAFRAME
    # =========================
    df = pd.DataFrame(ws.values)

    df = pd.DataFrame(ws.values)

    # hapus baris ganggu
    df = df.drop(index=[0, 2]).reset_index(drop=True)

    # set header dari baris pertama
    df.columns = df.iloc[0]

    # ambil data saja
    df = df[1:].reset_index(drop=True)
    return df

file_listB_cleaned = []

for f in file_listB:
    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):
            df = preprocess_excel(path)

            # simpan (nama file, dataframe)
            file_listB_cleaned.append((f, df))

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listB_cleaned)}")

Total berhasil: 125


In [ ]:
output_folder = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/B_cleaned'

for f, df in file_listB_cleaned:
    # ganti ekstensi jadi .xlsx
    new_name = os.path.splitext(f)[0] + ".xlsx"
    output_path = os.path.join(output_folder, new_name)

    df.to_excel(output_path, index=False)

    print(f"Saved: {new_name}")

Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2025.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2024.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2023.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2022.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2021.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2020.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2019.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2018.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2017.xlsx
Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2016.xlsx
Saved: Laju Indeks Implisit PDRB Seri 20

Build Time Series Tables for Line Chart

In [ ]:
def get_pdrb_value_and_varname(df):
    var_name = str(df.columns[1])  # header kolom ke-2

    # hapus tahun di akhir (misalnya " 2025")
    var_name = re.sub(r"\s*\d{4}$", "", var_name).strip()

    for _, row in df.iterrows():
        label = str(row.iloc[0]).upper()

        if "PRODUK DOMESTIK REGIONAL BRUTO" in label or "PDRB" in label:
            return row.iloc[1], var_name

    return None, var_name

def extract_title_year(filename):
    name = os.path.splitext(filename)[0]
    match = re.search(r'(\d{4})$', name)

    if not match:
        return None, None

    year = int(match.group(1))
    title = name[:match.start()].strip()

    return title, year


# grouping
groups = defaultdict(list)

for filename, df in file_listB_cleaned:
    title, year = extract_title_year(filename)

    if title:
        groups[title].append((year, df))


# build time series
output_folder = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/B_time_series'
os.makedirs(output_folder, exist_ok=True)

for title, items in groups.items():
    data = []
    var_name_final = None

    for year, df in items:
        value, var_name = get_pdrb_value_and_varname(df)

        if value is not None:
            data.append((year, value))
            var_name_final = var_name

    if not data:
        continue

    # urutkan tahun
    data.sort(key=lambda x: x[0])

    # buat dataframe
    ts_df = pd.DataFrame(data, columns=["Tahun", var_name_final])

    # range tahun
    years = [x[0] for x in data]
    min_year, max_year = min(years), max(years)

    # save
    output_name = f"{title} {min_year}-{max_year}.xlsx"
    output_path = os.path.join(output_folder, output_name)

    ts_df.to_excel(output_path, index=False)

    print(f"Saved: {output_name}")

Saved: Laju Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2014-2025.xlsx
Saved: Indeks Implisit PDRB Seri 2010 Menurut Lapangan Usaha Kabupaten Buleleng, 2014-2025.xlsx
Saved: Laju Pertumbuhan Y-on-Y PDRB Seri 2010 Atas Dasar Harga Konstan Menurut Lapangan Usaha Kabupaten Buleleng, 2014-2025.xlsx
Saved: Distribusi Persentase PDRB Seri 2010 Atas Dasar Harga Berlaku Menurut Lapangan Usaha Kabupaten Buleleng, 2014-2025.xlsx
Saved: PDRB Seri 2010 Atas Dasar Harga Konstan Menurut Lapangan Usaha Kabupaten Buleleng, 2014-2025.xlsx
Saved: PDRB Seri 2010 Atas Dasar Harga Berlaku Menurut Lapangan Usaha Kabupaten Buleleng, 2014-2025.xlsx
Saved: Laju Pertumbuhan Y-on-Y PDRB Seri 2010 Atas Dasar Harga Konstan Menurut Pengeluaran Kabupaten Buleleng, 2014-2025.xlsx
Saved: Distribusi Persentase PDRB Seri 2010 Atas Dasar Harga Berlaku Menurut Pengeluaran Kabupaten Buleleng, 2014-2025.xlsx
Saved: PDRB Seri 2010 Atas Dasar Harga Konstan Menurut Pengeluaran Kabupaten Buleleng, 

# Tabel Tipe C

In [ ]:
import os

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/C'
file_listC = os.listdir(folder_path)

print(f"Files and directories in '{folder_path}':")
for item in file_listC:
    print(item)

Files and directories in '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/C':
Inflasi Bulanan (month to month) Kota Singaraja, 2025.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2024.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2023.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2022.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2021.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2020.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2019.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2018.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2017.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2016.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2015.xlsx
Inflasi Bulanan (month to month) Kota Singaraja, 2014.xlsx
Inflasi y-o-y (year on year) Kota Singaraja, 2024.xlsx
Inflasi y-o-y (year on year) Kota Singaraja, 2025.xlsx
Indeks Harga Konsumen Kota Singaraja, 2025.xlsx
Indeks Harga Konsumen Kota Singaraja, 2024.xlsx
Indeks 

In [ ]:
import pandas as pd
from openpyxl import load_workbook

def preprocess_excel(file_path):
    wb = load_workbook(file_path)
    ws = wb.active

    # =========================
    # 1. UNMERGE + FILL
    # =========================
    merged_cells = list(ws.merged_cells.ranges)

    for merged in merged_cells:
        min_row, min_col, max_row, max_col = (
            merged.min_row, merged.min_col,
            merged.max_row, merged.max_col
        )

        value = ws.cell(row=min_row, column=min_col).value

        ws.unmerge_cells(str(merged))

        for r in range(min_row, max_row + 1):
            for c in range(min_col, max_col + 1):
                ws.cell(row=r, column=c).value = value

    # =========================
    # 2. GABUNG B2 + B3
    # =========================
    b2 = ws["B2"].value or ""
    b3 = ws["B3"].value or ""

    if isinstance(b3, float):
        b3 = int(b3)

    nama_kolom = f"{b2} {b3}".strip()

    # =========================
    # 3. KE DATAFRAME
    # =========================
    df = pd.DataFrame(ws.values)

    # =========================
    # 4. DROP KOLOM A & ATUR STRUKTUR
    # =========================
    df = df.iloc[:, 1:]  # drop kolom A

    # baris:
    # 0 = judul
    # 1 = tahun
    # 2 = header bulan
    # 3 = data

    header = df.iloc[3]      # Januari–Desember
    data = df.iloc[4:5]      # ambil baris data saja

    data.columns = header

    # =========================
    # 5. DROP TAHUNAN
    # =========================
    data = data.drop(columns=["Tahunan"], errors="ignore")

    # =========================
    # 6. LONG FORMAT
    # =========================
    df_long = data.melt(
        var_name="Bulan",
        value_name=nama_kolom
    ).reset_index(drop=True)

    return df_long


file_listC_cleaned = []

for f in file_listC:
    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):
            df_long = preprocess_excel(path)

            file_listC_cleaned.append((f, df_long))

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listC_cleaned)}")

Total berhasil: 26


In [ ]:
file_listC_cleaned[0]

('Inflasi Bulanan (month to month) Kota Singaraja, 2025.xlsx',
         Bulan Inflasi Bulanan (month to month) Kota Singaraja (Persen) 2025
 0     Januari                                              -0.53           
 1    Februari                                              -0.81           
 2       Maret                                               1.71           
 3       April                                                0.8           
 4         Mei                                              -0.28           
 5        Juni                                               0.37           
 6        Juli                                               0.42           
 7     Agustus                                              -0.56           
 8   September                                              -0.06           
 9     Oktober                                               0.28           
 10   November                                               0.47           
 11   Desembe

In [ ]:
output_folder = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/C_cleaned'

for f, df in file_listC_cleaned:
    # ganti ekstensi jadi .xlsx
    new_name = os.path.splitext(f)[0] + ".xlsx"
    output_path = os.path.join(output_folder, new_name)

    df.to_excel(output_path, index=False)

    print(f"Saved: {new_name}")

Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2025.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2024.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2023.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2022.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2021.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2020.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2019.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2018.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2017.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2016.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2015.xlsx
Saved: Inflasi Bulanan (month to month) Kota Singaraja, 2014.xlsx
Saved: Inflasi y-o-y (year on year) Kota Singaraja, 2024.xlsx
Saved: Inflasi y-o-y (year on year) Kota Singaraja, 2025.xlsx
Saved: Indeks Harga Konsumen Kota Singaraja, 2025.xlsx
Saved: Indeks Harga Konsumen 

# Tabel Tipe D

In [ ]:
import os

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/D'
file_listD = os.listdir(folder_path)

print(f"Files and directories in '{folder_path}':")
for item in file_listD:
    print(item)

Files and directories in '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/D':
Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Lapangan Usaha Kabupaten Buleleng, 2025.xlsx
Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Lapangan Usaha Kabupaten Buleleng, 2024.xlsx
Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Lapangan Usaha Kabupaten Buleleng, 2023.xlsx
Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Pengeluaran Kabupaten Buleleng, 2025.xlsx
Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Pengeluaran Kabupaten Buleleng, 2024.xlsx
Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Pengeluaran Kabupaten Buleleng, 2023.xlsx
Distribusi Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Lapangan Usaha Atas Dasar Harga Berlaku Kabupaten Buleleng, 2023.xlsx
Distribusi Produk Domest

In [ ]:
import pandas as pd
from openpyxl import load_workbook
import re

def preprocess_excel(file_path):
    wb = load_workbook(file_path)
    ws = wb.active

    # =========================
    # 1. AMBIL B2 + B3 (UNTUK NAMA FILE)
    # =========================
    b2 = ws["B2"].value or ""
    b3 = ws["B3"].value or ""

    if isinstance(b3, float):
        b3 = int(b3)

    judul = f"{b2}, {b3}".strip()

    # =========================
    # 2. UNMERGE + PINDAH KE BAWAH
    # =========================
    for merged in list(ws.merged_cells.ranges):
        min_row, min_col, max_row, max_col = (
            merged.min_row, merged.min_col,
            merged.max_row, merged.max_col
        )

        value = ws.cell(row=min_row, column=min_col).value
        ws.unmerge_cells(str(merged))

        for r in range(min_row, max_row + 1):
            for c in range(min_col, max_col + 1):
                ws.cell(row=r, column=c).value = None

        ws.cell(row=max_row, column=min_col).value = value

    # =========================
    # 3. HAPUS ROW 2 & 3
    # =========================
    ws.delete_rows(2, 2)

    # =========================
    # 4. KE DATAFRAME
    # =========================
    df = pd.DataFrame(ws.values)

    # =========================
    # 5. AUTO DETECT HEADER
    # =========================
    header_row = None
    for i in range(5):
        row_values = df.iloc[i].astype(str).str.lower().tolist()

        if any("triwulan" in v for v in row_values):
            header_row = i
            break

    if header_row is None:
        raise ValueError("Header tidak ditemukan")

    df.columns = df.iloc[header_row]
    df = df.iloc[header_row + 1:].reset_index(drop=True)

    # =========================
    # 6. RENAME KOLOM PERTAMA
    # =========================
    df = df.rename(columns={df.columns[0]: "Periode"})

    # =========================
    # 7. TRANSPOSE
    # =========================
    df_t = df.set_index("Periode").T

    # =========================
    # 8. HAPUS BARIS JUMLAH
    # =========================
    df_t = df_t[df_t.index.str.strip().str.lower() != "jumlah"]
    df_t = df_t.drop(columns=["Produk Domestik Regional Bruto"], errors="ignore")

    # =========================
    # 9. CLEAN
    # =========================
    df_t.columns.name = None
    df_t.index.name = None

    return df_t, judul

In [ ]:
file_listD_cleaned = []

for f in file_listD:
    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):
            df, judul = preprocess_excel(path)

            file_listD_cleaned.append({
                "file": f,
                "data": df,
                "judul": judul
            })

            print(f"Sukses: {f}")

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listD_cleaned)}")

Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Lapangan Usaha Kabupaten Buleleng, 2025.xlsx
Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Lapangan Usaha Kabupaten Buleleng, 2024.xlsx
Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Lapangan Usaha Kabupaten Buleleng, 2023.xlsx
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Pengeluaran Kabupaten Buleleng, 2025.xlsx
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Pengeluaran Kabupaten Buleleng, 2024.xlsx
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Pengeluaran Kabupaten Buleleng, 2023.xlsx
Sukses: Distribusi Produk Domestik Regional Bruto (PDRB) Triwulanan Menurut Lapangan Usaha Atas Dasar Harga Berlaku Kabupaten Buleleng, 2023.xlsx
Sukses: Distribusi Produk Domestik Regional Bru

For Single Line Chart

In [ ]:
import pandas as pd
from openpyxl import load_workbook
import re

def preprocess_excel2(file_path):
    wb = load_workbook(file_path)
    ws = wb.active

    # =========================
    # 1. AMBIL B2 + B3
    # =========================
    b2 = ws["B2"].value or ""
    b3 = ws["B3"].value or ""

    if isinstance(b3, float):
        b3 = int(b3)

    judul = f"{b2}, {b3}".strip()

    # =========================
    # 2. UNMERGE + PINDAH
    # =========================
    for merged in list(ws.merged_cells.ranges):
        min_row, min_col, max_row, max_col = (
            merged.min_row, merged.min_col,
            merged.max_row, merged.max_col
        )

        value = ws.cell(row=min_row, column=min_col).value
        ws.unmerge_cells(str(merged))

        for r in range(min_row, max_row + 1):
            for c in range(min_col, max_col + 1):
                ws.cell(row=r, column=c).value = None

        ws.cell(row=max_row, column=min_col).value = value

    # =========================
    # 3. HAPUS ROW 2 & 3
    # =========================
    ws.delete_rows(2, 2)

    # =========================
    # 4. KE DATAFRAME
    # =========================
    df = pd.DataFrame(ws.values)

    # =========================
    # 5. DETECT HEADER
    # =========================
    header_row = None
    for i in range(5):
        row_values = df.iloc[i].astype(str).str.lower().tolist()

        if any("triwulan" in v for v in row_values):
            header_row = i
            break

    if header_row is None:
        raise ValueError("Header tidak ditemukan")

    df.columns = df.iloc[header_row]
    df = df.iloc[header_row + 1:].reset_index(drop=True)

    # =========================
    # 6. RENAME KOLOM PERTAMA
    # =========================
    df = df.rename(columns={df.columns[0]: "Periode"})

    # =========================
    # 7. TRANSPOSE
    # =========================
    df_t = df.set_index("Periode").T

    # =========================
    # 8. HAPUS JUMLAH
    # =========================
    df_t = df_t[df_t.index.astype(str).str.strip().str.lower() != "jumlah"]

    # =========================
    # 9. AMBIL PDRB SAJA
    # =========================
    col_pdrb = [c for c in df_t.columns if "produk domestik regional bruto" in str(c).lower()]

    if not col_pdrb:
        raise ValueError("Kolom PDRB tidak ditemukan")

    col_pdrb = col_pdrb[0]
    df_t = df_t[[col_pdrb]]

    # =========================
    # 10. RESET INDEX
    # =========================
    df_t = df_t.reset_index()
    df_t.columns = ["Periode", "value"]   # 🔥 paksa reset kolom (INI KUNCI FIX)

    # =========================
    # 11. BERSIHKAN JUDUL
    # =========================
    judul_bersih = judul
    judul_bersih = re.sub(r"Menurut.*?(?=Kabupaten|Kota)", "", judul_bersih, flags=re.IGNORECASE)
    judul_bersih = " ".join(judul_bersih.split())

    # =========================
    # 12. RENAME KOLOM VALUE
    # =========================
    df_t = df_t.rename(columns={"value": judul_bersih})

    return df_t, judul_bersih

In [ ]:
file_listD_cleaned2 = []

for f in file_listD:
    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):
            df, judul_bersih = preprocess_excel2(path)

            file_listD_cleaned2.append({
                "file": judul_bersih + ".xlsx",   # ✅ pakai judul bersih
                "data": df,
                "judul": judul_bersih
            })

            print(f"Sukses: {judul_bersih}")

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listD_cleaned2)}")

Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2025
Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2024
Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2023
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2025
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2024
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2023
Sukses: Distribusi Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2023
Sukses: Distribusi Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2024
Sukses: Distribusi Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen

In [ ]:
file_listD_cleaned2[0]

{'file': 'Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2025.xlsx',
 'data':         Periode  \
 0    Triwulan I   
 1   Triwulan II   
 2  Triwulan III   
 3   Triwulan IV   
 
   Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2025  
 0                                               5.64                                                            
 1                                                5.6                                                            
 2                                               5.64                                                            
 3                                               5.28                                                            ,
 'judul': 'Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2025'}

In [ ]:
output_folder = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/D_cleaned2'
os.makedirs(output_folder, exist_ok=True)

import re

def clean_filename(text):
    return re.sub(r'[\\/*?:"<>|]', "", text).strip()

file_listD_cleaned2 = []

for f in file_listD:
    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):
            df, judul_bersih = preprocess_excel2(path)

            safe_name = clean_filename(judul_bersih)[:150]
            save_path = os.path.join(output_folder, safe_name + ".xlsx")

            df.to_excel(save_path, index=False)

            file_listD_cleaned2.append({
                "file": safe_name + ".xlsx",
                "data": df,
                "judul": judul_bersih
            })

            print(f"Sukses: {safe_name}")

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listD_cleaned2)}")

Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2025
Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2024
Sukses: Laju Pertumbuhan (y-on-y) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2023
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2025
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2024
Sukses: Laju Pertumbuhan (q-to-q) Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2023
Sukses: Distribusi Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2023
Sukses: Distribusi Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen), 2024
Sukses: Distribusi Produk Domestik Regional Bruto (PDRB) Triwulanan Kabupaten Buleleng (Persen

# Tabel Tipe E

In [ ]:
import os

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/E'
file_listE = os.listdir(folder_path)

print(f"Files and directories in '{folder_path}':")
for item in file_listE:
    print(item)

Files and directories in '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/E':
Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2020.csv
Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2022.csv
Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2021.csv
Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2017.csv
Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2018.csv
Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (ribu ha), 2020.csv
Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2024.xlsx
Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2021.csv
Luas Panen Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2022.csv
Luas Pane

In [ ]:
import pandas as pd
import os

for file_name in file_listE:
    file_path = os.path.join(folder_path, file_name)

    if file_name.endswith(".csv"):
        try:
            # baca csv
            try:
                df = pd.read_csv(file_path)
            except:
                df = pd.read_csv(file_path, sep=';')

            # kata yang ingin dihapus
            remove_words = ['luas panen', 'luas areal -', 'produksi', 'perkebunan - ']

            # bersihkan nama kolom
            df.columns = (
                df.columns
                .str.lower()
                .str.replace('|'.join(remove_words), '', regex=True)
                .str.replace(r'\s+', ' ', regex=True)
                .str.strip()
                .str.replace(r'\s*\([^)]*\)$', '', regex=True)  # hapus kata terakhir dalam ()
                .str.title()
            )

            # nama file baru
            new_name = os.path.splitext(file_name)[0] + ".xlsx"
            new_path = os.path.join(folder_path, new_name)

            df.to_excel(new_path, index=False)

            print(f"Converted: {file_name} → {new_name}")

        except Exception as e:
            print(f"Gagal convert {file_name}: {e}")

Converted: Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2020.csv → Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2020.xlsx
Converted: Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2022.csv → Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2022.xlsx
Converted: Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2021.csv → Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2021.xlsx
Converted: Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2017.csv → Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2017.xlsx
Converted: Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2018.csv →

In [ ]:
import re
from collections import defaultdict

def normalize_filename(filename):
    temp = filename

    # hapus "di Kabupaten Buleleng"
    temp = re.sub(r"di Kabupaten Buleleng", "", temp, flags=re.IGNORECASE)

    # hapus teks dalam tanda kurung (termasuk kurungnya)
    temp = re.sub(r"\(.*?\)", "", temp)

    # rapikan spasi
    temp = " ".join(temp.split())

    return temp.strip()

# grouping berdasarkan nama yang sudah dinormalisasi
groups = defaultdict(list)

for f in file_listE:
    norm = normalize_filename(f)
    groups[norm].append(f)

# cek dan tampilkan hasil
found_duplicate = False

for norm, files in groups.items():
    if len(files) > 1:
        if not found_duplicate:
            print("Duplikat ditemukan:\n")
        found_duplicate = True

        print(f"Group: {norm}")
        for f in files:
            print(f"- {f}")
        print()

if not found_duplicate:
    print("Tidak ada duplikat.")

Tidak ada duplikat.


In [ ]:
import pandas as pd
import numpy as np
import os
import re

folder_path = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/E'
output_folder = '/content/drive/My Drive/STIS/SMT 7/SKRIPSI/TABEL/E_cleaned'

file_listE = os.listdir(folder_path)

kecamatan_target = [
    "Gerokgak","Seririt","Busungbiu","Banjar","Sukasada",
    "Buleleng","Sawan","Kubutambahan","Tejakula"
]

'''
membersihkan nilai (-, ..., kosong → 0)
memastikan numeric
drop komoditas yang semua 0
drop kecamatan yang semua 0
hanya pakai 9 kecamatan target
pilih komoditas yang ada di semua kecamatan
maksimal 5 komoditas
bersihkan nama komoditas:
hapus satuan di dataframe
jika ada / → ambil kata pertama
rename file sesuai komoditas
skip file jika tidak ada komoditas valid
simpan ke E_cleaned
'''

def preprocess_excel(file_path, original_filename):

    # =========================
    # 1. BACA EXCEL
    # =========================
    df = pd.read_excel(file_path)

    # =========================
    # 2. BERSIHKAN NAMA KOLOM
    # =========================
    remove_words = ['luas panen', 'luas areal -', 'produksi', 'perkebunan - ']

    df.columns = (
        df.columns
        .astype(str)
        .str.lower()
        .str.replace('|'.join(remove_words), '', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .str.title()
    )

    # =========================
    # 3. NORMALISASI NILAI
    # =========================
    df = df.replace(["-", "..."], 0)
    df = df.fillna(0)

    # =========================
    # 4. PASTIKAN NUMERIC
    # =========================
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # =========================
    # 5. DROP KOLOM SEMUA 0
    # =========================
    komoditas_cols = df.columns[1:]
    keep_cols = [c for c in komoditas_cols if (df[c] != 0).any()]
    df = df[[df.columns[0]] + keep_cols]

    # =========================
    # 6. DROP ROW SEMUA 0
    # =========================
    df = df[(df.iloc[:,1:] != 0).any(axis=1)]

    # =========================
    # 7. HAPUS BARIS TERAKHIR JIKA BULELENG
    # =========================
    last_row_name = str(df.iloc[-1,0]).strip().lower()

    if "buleleng" in last_row_name:
        df = df.iloc[:-1].reset_index(drop=True)

    # =========================
    # 8. AMBIL HANYA 9 KECAMATAN TARGET
    # =========================
    df.iloc[:,0] = df.iloc[:,0].astype(str).str.strip()
    df = df[df.iloc[:,0].isin(kecamatan_target)]

    if len(df) != 9:
        return None, None

    # =========================
    # 9. PILIH KOMODITAS YANG ADA DI SEMUA KECAMATAN
    # =========================
    komoditas_cols = df.columns[1:]

    selected_cols = [
        c for c in komoditas_cols
        if (df[c] > 0).all()
    ][:5]

    if len(selected_cols) == 0:
        return None, None

    # =========================
    # 10. DF FINAL
    # =========================
    df_final = df[[df.columns[0]] + selected_cols]

    # =========================
    # 11. HAPUS SATUAN DI DF
    # =========================
    clean_cols = [df.columns[0]]

    for c in selected_cols:
        # hapus satuan
        col = re.sub(r"\s*\([^)]*\)", "", c)

        # jika ada "/", ambil kata pertama
        col = col.split("/")[0]

        clean_cols.append(col.strip())

    df_final.columns = clean_cols

    # =========================
    # 12. BANGUN NAMA KOMODITAS
    # =========================
    clean_selected = []

    for c in selected_cols:
        # ambil kata sebelum "/"
        base = c.split("/")[0].strip()
        clean_selected.append(base)

    komoditas_string = ", ".join(clean_selected)

    name, ext = os.path.splitext(original_filename)

    # tambahkan komoditas sebelum kata "Menurut"
    new_name = re.sub(
        r"(.*?)\s+Menurut",
        r"\1 " + komoditas_string + " Menurut Kecamatan",
        name,
        count=1,
        flags=re.IGNORECASE
    )

    # hapus "dan Jenis Tanaman"
    new_name = new_name.replace("Jenis Tanaman", "")

    # rapikan spasi
    new_name = re.sub(r"\s+", " ", new_name).strip()

    new_filename = new_name + ext

    return df_final, new_filename

file_listE_cleaned = []

for f in file_listE:
    path = os.path.join(folder_path, f)

    try:
        if f.endswith(".xlsx"):
            df, new_name = preprocess_excel(path, f)

            if df is None:
                print(f"Skip (tidak ada komoditas valid): {f}")
                continue

            save_path = os.path.join(output_folder, new_name)
            df.to_excel(save_path, index=False)

            file_listE_cleaned.append((new_name, df))

            print(f"Sukses: {f} -> {new_name}")

    except Exception as e:
        print(f"Gagal: {f} - {e}")

print(f"Total berhasil: {len(file_listE_cleaned)}")

Skip (tidak ada komoditas valid): Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2024.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Hias Menurut Kecamatan dan Jenis Tanaman  di Kabupaten Buleleng, 2025.xlsx
Sukses: Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2025.xlsx -> Produksi Tanaman Sayuran Cabai Rawit (Kuintal) (Kw) Menurut Kecamatan di Kabupaten Buleleng, 2025.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Hias Menurut Kecamatan dan Jenis Tanaman  di Kabupaten Buleleng, 2024.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2020.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2022.xlsx
Sukses: Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2021.xlsx -> Luas Areal Tanaman Perkebu

/tmp/ipykernel_3234/2659016738.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(["-", "..."], 0)


Sukses: Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2017.xlsx -> Luas Areal Tanaman Perkebunan Kelapa (Ribu Ha), Kakao (Ribu Ha) Menurut Kecamatan di Kabupaten Buleleng (Ribu ha), 2017.xlsx
Sukses: Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2018.xlsx -> Luas Areal Tanaman Perkebunan Kelapa (Ribu Ha), Kakao (Ribu Ha) Menurut Kecamatan di Kabupaten Buleleng (Ribu ha), 2018.xlsx
Sukses: Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (ribu ha), 2020.xlsx -> Luas Areal Tanaman Perkebunan Kelapa (Ribu Ha), Kakao (Ribu Ha) Menurut Kecamatan di Kabupaten Buleleng (ribu ha), 2020.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2021.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2022.xlsx
Skip 

/tmp/ipykernel_3234/2659016738.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(["-", "..."], 0)
/tmp/ipykernel_3234/2659016738.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(["-", "..."], 0)
/tmp/ipykernel_3234/2659016738.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasti

Skip (tidak ada komoditas valid): Luas Panen Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman  di Kabupaten Buleleng, 2024.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman  di Kabupaten Buleleng, 2024.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2022.xlsx
Sukses: Produksi Buah-buahan Menurut Jenis Tanaman Menurut Kecamatan di Kabupaten Buleleng, 2019.xlsx -> Produksi Buah-buahan Mangga (Kuintal), Nangka, Pepaya (Kuintal), Pisang (Kuintal), Rambutan (Kuintal) Menurut Jenis Tanaman Menurut Kecamatan di Kabupaten Buleleng, 2019.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2019.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Biofarmaka Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2018.xlsx
Sukses: Produksi Buah-buahan Menurut Jenis Tanaman Menurut Keca

...

Sukses: Produksi Buah-buahan Menurut Jenis Tanaman Menurut Kecamatan di Kabupaten Buleleng, 2024.xlsx -> Produksi Buah-buahan Durian (Kuintal), Mangga (Kuintal), Nangka, Pepaya (Kuintal), Pisang (Kuintal) Menurut Jenis Tanaman Menurut Kecamatan di Kabupaten Buleleng, 2024.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2018.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2018.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2023.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2019.xlsx


/tmp/ipykernel_3234/2659016738.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(["-", "..."], 0)


Skip (tidak ada komoditas valid): Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2022.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2020.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2021.xlsx
Skip (tidak ada komoditas valid): Produksi Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2021.xlsx


/tmp/ipykernel_3234/2659016738.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(["-", "..."], 0)
/tmp/ipykernel_3234/2659016738.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(["-", "..."], 0)
/tmp/ipykernel_3234/2659016738.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasti

Sukses: Produksi Tanaman Sayuran Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2023.xlsx -> Produksi Tanaman Sayuran Cabai Rawit (Kuintal) Menurut Kecamatan di Kabupaten Buleleng, 2023.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2018.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2021.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2023.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2024.xlsx
Skip (tidak ada komoditas valid): Luas Panen Tanaman Hias Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng, 2020.xlsx
Sukses: Luas Areal Tanaman Perkebunan Menurut Kecamatan dan Jenis Tanaman di Kabupaten Buleleng (Ribu ha), 2019.xlsx -> Luas Areal Tanaman Perkebunan Kelapa (Ribu

In [ ]:
file_listE_cleaned[8]

('Produksi Buah-buahan Menurut Jenis Tanaman Menurut Kecamatan di Kabupaten Buleleng, 2023.xlsx',
       Kecamatan    Durian    Mangga  Nangka/Cempedak    Pisang  Rambutan
 0      Gerokgak      90.0   74798.0           3458.0  105576.0    7975.0
 1       Seririt    7458.0   75291.0            639.0   40959.0    2879.0
 2     Busungbiu   10139.0    2852.0            162.0    5778.0    2197.0
 3        Banjar   20830.0    9924.0            452.0     265.0    4684.0
 4      Sukasada    2428.0    3751.0            110.0    7118.0    8611.0
 5      Buleleng     583.0    8865.0            483.7   18175.9   11796.5
 6         Sawan    6494.0   11749.0             72.0    7538.0   11190.0
 7  Kubutambahan  103745.6  105248.0           2008.0   11786.0   28952.4
 8      Tejakula    8919.0   26082.0           4389.0   46673.6   43426.2)